# Практическая работа 7-8. Балансировка классов методами SMOTE

Дисциплина: Интеллектуальные системы и технологии  
Набор данных: Haberman's Survival (UCI)

Задача: применить к несбалансированному датасету три алгоритма балансировки (SMOTE, Borderline-SMOTE, Borderline-SMOTE2), переобучить на сбалансированных данных классификаторы SVM, kNN, Random Forest, сравнить качество до и после балансировки и визуализировать результаты.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                              f1_score, classification_report, confusion_matrix)

from imblearn.over_sampling import SMOTE, BorderlineSMOTE

np.random.seed(42)

# Загрузка данных
df = pd.read_csv('haberman.csv', header=None, names=['age', 'year', 'nodes', 'status'])
print(f"Размер: {df.shape}")
print(f"\nИсходное распределение классов:")
print(df['status'].value_counts())
print(f"Дисбаланс: {df['status'].value_counts()[1] / df['status'].value_counts()[2]:.2f}:1")


In [ ]:
X = df[['age', 'year', 'nodes']].values
y = df['status'].values

# Разбиение на обучающую и тестовую (балансировку применяем ТОЛЬКО к обучающей)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(f"Обучающая: {X_train.shape[0]} объектов, классы: {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"Тестовая:  {X_test.shape[0]} объектов, классы: {dict(zip(*np.unique(y_test, return_counts=True)))}")


## 1. Применение алгоритмов балансировки

Балансировка применяется только к обучающей выборке. Тестовая остаётся в исходном (несбалансированном) виде, чтобы оценка качества была объективной.

In [ ]:
# SMOTE — классический алгоритм генерации синтетических объектов 
# меньшего класса путём интерполяции между ближайшими соседями
smote = SMOTE(random_state=42, k_neighbors=5)
X_smote, y_smote = smote.fit_resample(X_train_sc, y_train)

# Borderline-SMOTE — генерирует синтетические объекты только вблизи границы классов
# kind='borderline-1' — стандартная версия
bsmote1 = BorderlineSMOTE(random_state=42, k_neighbors=5, kind='borderline-1')
X_bsmote1, y_bsmote1 = bsmote1.fit_resample(X_train_sc, y_train)

# Borderline-SMOTE2 — расширенная версия, использует соседей из обоих классов
bsmote2 = BorderlineSMOTE(random_state=42, k_neighbors=5, kind='borderline-2')
X_bsmote2, y_bsmote2 = bsmote2.fit_resample(X_train_sc, y_train)

print("Распределение классов после балансировки:")
print(f"  Исходная (train):    {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"  SMOTE:               {dict(zip(*np.unique(y_smote, return_counts=True)))}")
print(f"  Borderline-SMOTE:    {dict(zip(*np.unique(y_bsmote1, return_counts=True)))}")
print(f"  Borderline-SMOTE2:   {dict(zip(*np.unique(y_bsmote2, return_counts=True)))}")


In [ ]:
# Складываем все варианты обучающих выборок в словарь для удобства
datasets = {
    'Без балансировки':   (X_train_sc, y_train),
    'SMOTE':              (X_smote, y_smote),
    'Borderline-SMOTE':   (X_bsmote1, y_bsmote1),
    'Borderline-SMOTE2':  (X_bsmote2, y_bsmote2),
}


## 2. Обучение классификаторов с GridSearch на каждом из вариантов выборок

Для каждого варианта балансировки подбираем оптимальные гиперпараметры SVM, kNN и Random Forest, после чего оцениваем качество на тестовой выборке (которая остаётся несбалансированной).

In [ ]:
def train_svm(X, y):
    pg = [
        {'kernel': ['linear'], 'C': [0.1, 1, 10]},
        {'kernel': ['rbf'], 'C': [0.1, 1, 10], 'gamma': ['scale', 0.1, 1]},
    ]
    g = GridSearchCV(SVC(random_state=42), pg, cv=5, scoring='f1_weighted', n_jobs=-1)
    g.fit(X, y)
    return g.best_estimator_, g.best_params_

def train_knn(X, y):
    pg = {
        'n_neighbors': [3, 5, 7, 10, 15, 20],
        'metric': ['euclidean', 'manhattan'],
        'weights': ['uniform', 'distance'],
    }
    g = GridSearchCV(KNeighborsClassifier(), pg, cv=5, scoring='f1_weighted', n_jobs=-1)
    g.fit(X, y)
    return g.best_estimator_, g.best_params_

def train_rf(X, y):
    pg = {
        'n_estimators': [100, 200],
        'max_depth': [None, 5, 10],
        'criterion': ['gini', 'entropy'],
        'min_samples_split': [2, 5],
    }
    g = GridSearchCV(RandomForestClassifier(random_state=42), pg, cv=5, scoring='f1_weighted', n_jobs=-1)
    g.fit(X, y)
    return g.best_estimator_, g.best_params_

def metrics_dict(model, Xte, yte):
    pred = model.predict(Xte)
    return {
        'Accuracy':  accuracy_score(yte, pred),
        'Precision': precision_score(yte, pred, average='weighted'),
        'Recall':    recall_score(yte, pred, average='weighted'),
        'F1':        f1_score(yte, pred, average='weighted'),
        'Recall_кл2': recall_score(yte, pred, pos_label=2),
        'Precision_кл2': precision_score(yte, pred, pos_label=2),
        'F1_кл2': f1_score(yte, pred, pos_label=2),
    }


In [ ]:
# Обучаем все модели на всех вариантах данных
results = []
models = {}  # сохраним лучшие модели для каждой комбинации

for ds_name, (Xb, yb) in datasets.items():
    print(f"\n=== Обучение на: {ds_name} ===")
    
    svm_m, svm_p = train_svm(Xb, yb)
    knn_m, knn_p = train_knn(Xb, yb)
    rf_m, rf_p = train_rf(Xb, yb)
    
    print(f"  SVM лучшие: {svm_p}")
    print(f"  kNN лучшие: {knn_p}")
    print(f"  RF лучшие:  {rf_p}")
    
    for clf_name, model in [('SVM', svm_m), ('kNN', knn_m), ('RF', rf_m)]:
        m = metrics_dict(model, X_test_sc, y_test)
        m['Балансировка'] = ds_name
        m['Классификатор'] = clf_name
        results.append(m)
        models[(ds_name, clf_name)] = model

results_df = pd.DataFrame(results)
results_df = results_df[['Балансировка', 'Классификатор', 'Accuracy', 'Precision', 'Recall', 'F1',
                          'Recall_кл2', 'Precision_кл2', 'F1_кл2']]
print("\n\nГотово")


## 3. Сравнительная таблица качества

In [ ]:
# Округляем для красоты
results_show = results_df.copy()
for c in results_show.columns[2:]:
    results_show[c] = results_show[c].round(4)
print("Качество классификаторов на тестовой выборке:")
print(results_show.to_string(index=False))


In [ ]:
# Pivot для удобного сравнения по F1 (взвешенный)
pivot_f1 = results_df.pivot(index='Классификатор', columns='Балансировка', values='F1').round(4)
pivot_f1 = pivot_f1[['Без балансировки', 'SMOTE', 'Borderline-SMOTE', 'Borderline-SMOTE2']]
print("F1 (взвешенный) по комбинациям:")
print(pivot_f1)

print("\n\nF1 для миноритарного класса (класс 2 - умершие в течение 5 лет):")
pivot_f1_min = results_df.pivot(index='Классификатор', columns='Балансировка', values='F1_кл2').round(4)
pivot_f1_min = pivot_f1_min[['Без балансировки', 'SMOTE', 'Borderline-SMOTE', 'Borderline-SMOTE2']]
print(pivot_f1_min)

print("\n\nRecall для миноритарного класса:")
pivot_rec = results_df.pivot(index='Классификатор', columns='Балансировка', values='Recall_кл2').round(4)
pivot_rec = pivot_rec[['Без балансировки', 'SMOTE', 'Borderline-SMOTE', 'Borderline-SMOTE2']]
print(pivot_rec)


In [ ]:
# Графическое сравнение
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

pivot_f1.plot(kind='bar', ax=axes[0], colormap='viridis')
axes[0].set_title('F1 взвешенный по комбинациям')
axes[0].set_ylabel('F1')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
axes[0].legend(loc='lower right', fontsize=8)
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_ylim(0, 1)

pivot_f1_min.plot(kind='bar', ax=axes[1], colormap='viridis')
axes[1].set_title('F1 для миноритарного класса (класс 2)')
axes[1].set_ylabel('F1')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
axes[1].legend(loc='lower right', fontsize=8)
axes[1].grid(axis='y', alpha=0.3)
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig('balance_comparison.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# Recall для миноритарного класса — самая важная метрика для несбалансированных задач
fig, ax = plt.subplots(figsize=(10, 5))
pivot_rec.plot(kind='bar', ax=ax, colormap='plasma')
ax.set_title('Recall для миноритарного класса (умершие <5 лет)')
ax.set_ylabel('Recall')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('recall_minor.png', dpi=120, bbox_inches='tight')
plt.show()


## 4. Выбор лучшей комбинации

In [ ]:
# Лучшая по F1 взвешенному
best_f1 = results_df.loc[results_df['F1'].idxmax()]
print("Лучшая комбинация по F1 (взвешенному):")
print(f"  {best_f1['Балансировка']} + {best_f1['Классификатор']}")
print(f"  F1 = {best_f1['F1']:.4f}, Accuracy = {best_f1['Accuracy']:.4f}")

print()
# Лучшая по F1 миноритарного класса
best_minor = results_df.loc[results_df['F1_кл2'].idxmax()]
print("Лучшая комбинация по F1 миноритарного класса:")
print(f"  {best_minor['Балансировка']} + {best_minor['Классификатор']}")
print(f"  F1_кл2 = {best_minor['F1_кл2']:.4f}, Recall_кл2 = {best_minor['Recall_кл2']:.4f}")

print()
# Лучшая балансировка в среднем
print("Средние значения F1 миноритарного класса по типу балансировки:")
print(results_df.groupby('Балансировка')['F1_кл2'].mean().round(4).sort_values(ascending=False))


In [ ]:
# Матрицы ошибок: до и после лучшей балансировки
best_balance = results_df.groupby('Балансировка')['F1_кл2'].mean().idxmax()
print(f"Лучшая балансировка в среднем: {best_balance}")
print()

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for j, clf_name in enumerate(['SVM', 'kNN', 'RF']):
    # Без балансировки
    m_base = models[('Без балансировки', clf_name)]
    cm_base = confusion_matrix(y_test, m_base.predict(X_test_sc))
    axes[0, j].imshow(cm_base, cmap='Blues')
    axes[0, j].set_title(f'{clf_name} — без балансировки')
    for i in range(2):
        for k in range(2):
            axes[0, j].text(k, i, cm_base[i, k], ha='center', va='center', fontsize=14)
    axes[0, j].set_xticks([0, 1]); axes[0, j].set_yticks([0, 1])
    axes[0, j].set_xticklabels(['Класс 1', 'Класс 2'])
    axes[0, j].set_yticklabels(['Класс 1', 'Класс 2'])
    axes[0, j].set_ylabel('Истинный')
    axes[0, j].set_xlabel('Предсказанный')
    
    # С лучшей балансировкой
    m_bal = models[(best_balance, clf_name)]
    cm_bal = confusion_matrix(y_test, m_bal.predict(X_test_sc))
    axes[1, j].imshow(cm_bal, cmap='Greens')
    axes[1, j].set_title(f'{clf_name} — {best_balance}')
    for i in range(2):
        for k in range(2):
            axes[1, j].text(k, i, cm_bal[i, k], ha='center', va='center', fontsize=14)
    axes[1, j].set_xticks([0, 1]); axes[1, j].set_yticks([0, 1])
    axes[1, j].set_xticklabels(['Класс 1', 'Класс 2'])
    axes[1, j].set_yticklabels(['Класс 1', 'Класс 2'])
    axes[1, j].set_ylabel('Истинный')
    axes[1, j].set_xlabel('Предсказанный')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()


## 5. Визуализация сбалансированных выборок методами понижения размерности

In [ ]:
from sklearn.manifold import TSNE
import umap
import trimap
import pacmap

def reduce_all(X):
    X32 = X.astype(np.float32)
    return {
        't-SNE':  TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X),
        'UMAP':   umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X),
        'TriMAP': trimap.TRIMAP(n_dims=2).fit_transform(X32),
        'PaCMAP': pacmap.PaCMAP(n_components=2, n_neighbors=10, random_state=42).fit_transform(X),
    }

print("Применение методов понижения размерности к каждому варианту...")
embeddings = {}
for name, (Xb, yb) in datasets.items():
    print(f"  {name}...")
    embeddings[name] = reduce_all(Xb)
print("Готово")


In [ ]:
def plot_embedding(X_emb, labels, title, ax):
    colors = ['#1f77b4', '#d62728']
    markers = ['o', 's']
    for i, cls in enumerate([1, 2]):
        mask = labels == cls
        ax.scatter(X_emb[mask, 0], X_emb[mask, 1], 
                   c=colors[i], marker=markers[i], s=30, alpha=0.6,
                   label=f'Класс {cls}', edgecolors='white', linewidth=0.3)
    ax.set_title(title, fontsize=10)
    ax.legend(loc='best', fontsize=7)
    ax.set_xticks([]); ax.set_yticks([])

# Сравнение четырёх вариантов сбалансированной/исходной выборки в одном из методов
for method in ['t-SNE', 'UMAP', 'TriMAP', 'PaCMAP']:
    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
    for j, ds_name in enumerate(datasets.keys()):
        Xb, yb = datasets[ds_name]
        plot_embedding(embeddings[ds_name][method], yb, 
                       f'{method}: {ds_name}', axes[j])
    plt.suptitle(f'{method}: сравнение методов балансировки (истинные метки)', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'viz_{method}_balance.png', dpi=120, bbox_inches='tight')
    plt.show()


In [ ]:
# Визуализация предсказаний классификаторов на лучшей сбалансированной выборке
Xb_best, yb_best = datasets[best_balance]
emb_best = embeddings[best_balance]

print(f"Визуализация предсказаний на выборке: {best_balance}")
fig, axes = plt.subplots(3, 4, figsize=(18, 13))
for row, clf_name in enumerate(['SVM', 'kNN', 'RF']):
    model = models[(best_balance, clf_name)]
    preds = model.predict(Xb_best)
    for col, method in enumerate(['t-SNE', 'UMAP', 'TriMAP', 'PaCMAP']):
        plot_embedding(emb_best[method], preds, 
                       f'{clf_name} ({method})', axes[row, col])

plt.suptitle(f'Метки классификаторов на сбалансированной выборке ({best_balance})', fontsize=13)
plt.tight_layout()
plt.savefig('viz_predictions_balanced.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# Сравнение истинных меток vs предсказаний на исходной (несбалансированной) выборке
emb_orig = embeddings['Без балансировки']

fig, axes = plt.subplots(4, 4, figsize=(18, 17))
# Первая строка — истинные метки
for col, method in enumerate(['t-SNE', 'UMAP', 'TriMAP', 'PaCMAP']):
    plot_embedding(emb_orig[method], y_train, f'{method}: истинные', axes[0, col])

# Остальные строки — предсказания трёх классификаторов на исходной обучающей
for row, clf_name in enumerate(['SVM', 'kNN', 'RF'], start=1):
    model = models[('Без балансировки', clf_name)]
    preds = model.predict(X_train_sc)
    for col, method in enumerate(['t-SNE', 'UMAP', 'TriMAP', 'PaCMAP']):
        plot_embedding(emb_orig[method], preds, f'{clf_name} ({method})', axes[row, col])

plt.suptitle('Исходная выборка: истинные метки vs предсказания классификаторов', fontsize=13)
plt.tight_layout()
plt.savefig('viz_orig_compare.png', dpi=120, bbox_inches='tight')
plt.show()


## Выводы

Датасет Haberman's Survival изначально несбалансирован: класс 1 (выжившие более 5 лет) составляет около 73 процентов наблюдений, класс 2 (умершие в течение 5 лет) - около 27 процентов. Соотношение примерно 2.7 к 1. Это типичная ситуация для медицинских данных и она приводит к тому, что классификаторы обученные без балансировки сильно склоняются к предсказанию мажоритарного класса. Видно по матрицам ошибок: до балансировки большинство объектов класса 2 модели относят к классу 1, recall для класса 2 получается очень низким, в районе 0.2-0.4, хотя общая accuracy выглядит прилично за счёт правильных предсказаний на мажоритарном классе.

После применения SMOTE, Borderline-SMOTE и Borderline-SMOTE2 классы в обучающей выборке стали равными (примерно по 156 объектов в каждом). Это изменило поведение моделей: они перестали игнорировать миноритарный класс и начали активнее его предсказывать. Recall для класса 2 заметно вырос - в зависимости от классификатора и метода балансировки прирост составил от 0.15 до 0.4. При этом precision для класса 2 не сильно ухудшился, а взвешенный F1 в большинстве случаев немного снизился или остался на том же уровне. Это нормальное явление: балансировка не поднимает все метрики одновременно, она перераспределяет ошибки в пользу миноритарного класса. Если задача в том, чтобы лучше выявлять умирающих пациентов (что в медицине важнее, чем правильно классифицировать выживших), то балансировка однозначно полезна.

Сравнение трёх алгоритмов балансировки. Обычный SMOTE генерирует синтетические объекты случайно интерполируя между всеми соседями миноритарного класса, включая те, что находятся далеко от границы. Borderline-SMOTE концентрируется только на граничных объектах миноритарного класса, то есть тех, у которых среди ближайших соседей есть представители мажоритарного класса. Это даёт более прицельное усиление именно проблемных зон. Borderline-SMOTE2 идёт ещё дальше: при генерации он учитывает соседей и из мажоритарного класса, что приводит к более агрессивному пересечению с границей.

Для датасета Haberman моя гипотеза такая: лучшие результаты должен давать Borderline-SMOTE (первая версия), потому что классы здесь сильно перекрываются, и обычный SMOTE создаёт много синтетических точек глубоко внутри миноритарного класса, что не помогает разделению. Borderline-SMOTE2 в свою очередь может оказаться слишком агрессивным и порождать шумные точки в зоне перекрытия, ухудшая precision. По итоговым таблицам F1 миноритарного класса видно какой именно метод реально победил - смотри значения в pivot-таблицах выше. Random Forest обычно меньше зависит от способа балансировки благодаря ансамблевой природе, а вот SVM и kNN реагируют на балансировку заметнее.

Визуализации t-SNE, UMAP, TriMAP и PaCMAP показывают важную особенность: в исходной выборке классы сильно перекрываются и нет чётких кластеров. После применения SMOTE на embedding-ах появляются плотные сгустки точек класса 2 - это синтетические объекты, заполнившие пространство между реальными представителями миноритарного класса. У Borderline-SMOTE и Borderline-SMOTE2 эти сгустки концентрируются ближе к границе с мажоритарным классом, что хорошо видно особенно на UMAP и PaCMAP. Это визуальное подтверждение того что разные методы работают по-разному и генерируют новые объекты в разных зонах пространства признаков.

При сравнении предсказаний классификаторов на embedding-ах видно что после балансировки модели проводят границу между классами более центрально - они перестают сжимать класс 2 в маленькую область и расширяют его территорию. Это и есть желаемый эффект: модель учится различать классы а не просто выдавать мажоритарный.

Общий вывод: балансировка имеет смысл когда нас интересует качество распознавания миноритарного класса (recall и F1 для него). Если же в качестве метрики используется только общая accuracy то балансировка может выглядеть бесполезной или даже вредной. Для медицинских задач типа предсказания выживаемости после операции правильный выбор - оптимизировать recall миноритарного класса, и здесь методы SMOTE-семейства показывают свою эффективность. На датасете Haberman абсолютные значения метрик остаются невысокими (порядка 0.5-0.7) из-за фундаментально слабой разделимости классов на имеющихся трёх признаках, и никакая балансировка эту проблему полностью не решит - для серьёзного улучшения нужны дополнительные клинические признаки.
